# WavqWise: EEG Signal Classification
**Sense. Forecast. Alert.**

Classify EEG signals into mental states (Relaxed / Focused / Drowsy) using frequency band power features.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VK-Ant/wavqwise/blob/main/demos/notebooks/wavqwise_eeg_classification.ipynb)

**Author:** [VK-Ant](https://github.com/VK-Ant)

In [ ]:
# Install WavqWise
!pip install wavqwise -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, welch
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns

from wavqwise import SignalPipeline
print(f'WavqWise loaded')

## 1. Generate Synthetic EEG Dataset
3 classes simulating real clinical EEG patterns:
- **Relaxed**: Dominant alpha rhythm (8-13 Hz)
- **Focused**: Dominant beta rhythm (13-30 Hz)
- **Drowsy**: Dominant theta+delta (0.5-8 Hz)

In [ ]:
def generate_eeg_dataset(n_samples=300, duration=4, sample_rate=256):
    n_points = duration * sample_rate
    t = np.arange(n_points) / sample_rate
    data, labels = [], []

    for i in range(n_samples):
        label = i % 3
        noise = np.random.normal(0, 1.5, n_points)
        if label == 0:  # Relaxed
            signal = 8*np.sin(2*np.pi*10*t + np.random.uniform(0,2*np.pi)) + 2*np.sin(2*np.pi*6*t) + noise
        elif label == 1:  # Focused
            signal = 7*np.sin(2*np.pi*22*t + np.random.uniform(0,2*np.pi)) + 3*np.sin(2*np.pi*18*t) + noise
        else:  # Drowsy
            signal = 6*np.sin(2*np.pi*5*t + np.random.uniform(0,2*np.pi)) + 5*np.sin(2*np.pi*2*t) + noise
        data.append(signal)
        labels.append(label)

    return np.array(data), np.array(labels), t

signals, labels, t = generate_eeg_dataset(300, 4, 256)
label_names = {0: 'Relaxed', 1: 'Focused', 2: 'Drowsy'}
print(f'Dataset: {signals.shape[0]} epochs, {signals.shape[1]} samples each')

## 2. Visualize Sample Epochs

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 8))
for cls, (color, name) in enumerate(zip(['#059669','#2563eb','#d97706'], ['Relaxed','Focused','Drowsy'])):
    idx = np.where(labels == cls)[0][0]
    axes[cls, 0].plot(t[:512], signals[idx][:512], color=color, linewidth=0.8)
    axes[cls, 0].set_title(f'{name} - Time Domain')
    axes[cls, 0].set_ylabel('Amplitude')
    f, psd = welch(signals[idx], fs=256, nperseg=256)
    axes[cls, 1].semilogy(f[:60], psd[:60], color=color)
    axes[cls, 1].set_title(f'{name} - PSD')
    axes[cls, 1].axvspan(8,13,alpha=0.1,color='green')
    axes[cls, 1].axvspan(13,30,alpha=0.1,color='blue')
plt.tight_layout()
plt.show()

## 3. WavqWise SignalPipeline - Quick Band Analysis

In [ ]:
# Use WavqWise for single-epoch analysis
sig = SignalPipeline()
sig._data = signals[0]  # Relaxed epoch
sig._sample_rate = 256
sig.filter(low=1, high=50, notch=50)
bands = sig.extract_bands(['delta','theta','alpha','beta','gamma'])

print('Relaxed epoch band power:')
for name, power in bands.bands.items():
    print(f'  {name}: {power:.4f} uV^2')

## 4. Feature Extraction (All Epochs)

In [ ]:
def extract_features(signals, sr=256):
    band_defs = {'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,50)}
    features = []
    for signal in signals:
        b, a = butter(4, [0.5, 50], btype='band', fs=sr)
        filtered = filtfilt(b, a, signal)
        freqs, psd = welch(filtered, fs=sr, nperseg=min(256, len(filtered)))
        bp = {}
        total = 0
        for name, (lo, hi) in band_defs.items():
            mask = (freqs >= lo) & (freqs <= hi)
            power = np.trapezoid(psd[mask], freqs[mask]) if hasattr(np, 'trapezoid') else np.sum(psd[mask])
            bp[name] = power
            total += power
        for name in band_defs:
            bp[f'{name}_rel'] = bp[name] / max(total, 1e-8)
        bp['alpha_beta_ratio'] = bp['alpha'] / max(bp['beta'], 1e-8)
        bp['theta_alpha_ratio'] = bp['theta'] / max(bp['alpha'], 1e-8)
        features.append(bp)
    return pd.DataFrame(features)

features = extract_features(signals)
print(f'Feature matrix: {features.shape}')
features.head()

## 5. Train & Evaluate Classifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.3, random_state=42, stratify=labels)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.1%}')
print(classification_report(y_test, y_pred, target_names=['Relaxed','Focused','Drowsy']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Relaxed','Focused','Drowsy'], yticklabels=['Relaxed','Focused','Drowsy'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 6. Feature Importance

In [ ]:
imp = pd.Series(clf.feature_importances_, index=features.columns).nlargest(10)
plt.figure(figsize=(8,5))
imp.plot(kind='barh', color='#2563eb')
plt.title('Top 10 Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

---
**WavqWise** - Sense. Forecast. Alert. | [GitHub](https://github.com/VK-Ant/wavqwise) | [PyPI](https://pypi.org/project/wavqwise/)